A prototype with small amount of data to solely test on non built-in ctc algorithm. The loss function lacks the ability to vectorize efficiently so prototype can only run on small amount of data for testing the learning capability of the model.

In [52]:
import torch
import torchaudio
import torchaudio.transforms as T
import torch.nn as nn
from tokenizer import TokenizerBPE01
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import numpy as np
import pandas as pd
import pickle as pkl
import json

# CTC
from ctc_utils import *

# Models
from models import models

print("Training set up...")
# Reallocate hugging face cache
import os
from pathlib import Path
from datasets import load_dataset
from datasets import config
print("Huggingface cache location:", config.HF_DATASETS_CACHE)

Training set up...
Huggingface cache location: D:\workplace\hf_cache\datasets


In [53]:
LEARNING_RATE = 0.001
BATCH_SIZE = 32
EPOCHS = 3

TRAIN_RATIO = 0.005
VALIDATION_RATIO = 0.0025
BATCH_RATIO = 1

TOK_PATH = "vocab.json"
NORM_PATH = "normalizer1.pkl"

SEED = 42


In [54]:
device = torch.device("cpu")
torch.manual_seed(SEED)

In [55]:
dataset = load_dataset(
    "openslr/librispeech_asr",
    "clean",
    split="train.100"
)

test_ratio = 1 - TRAIN_RATIO - VALIDATION_RATIO

train_end = int(len(dataset) * TRAIN_RATIO)
validation_end = train_end + int(len(dataset) * VALIDATION_RATIO)

train_set = dataset[:train_end]
val_set = dataset[train_end:validation_end]
# test_set = dataset[validation_end:]
print("Train size     : ", train_end)
print("Validation size: ", validation_end - train_end)
print(f"Test set index : {validation_end}-{len(dataset)}")
print()
del dataset

Train size     :  142
Validation size:  71
Test set index : 213-28539



In [56]:
# Data preparation
def get_input_and_target(dataset):
    # Input output
    input_audios = dataset['audio']
    target_sentences = dataset['text']

    return input_audios, target_sentences


class SpeechDataset(Dataset):

    def __init__(self, input_audios, target_sentences, tokenizer, mfcc_transform, device):
        self.input_audios = input_audios
        self.target_sentences = target_sentences
        self.tokenizer = tokenizer
        self.mfcc_transform = mfcc_transform
        self.device = device

    def __len__(self):
        return len(self.input_audios)

    def __getitem__(self, index):
        # Preprocess input
        try:
            input_audio = torch.tensor(self.input_audios[index])
        except:
            # For codec
            input_audio = torch.tensor(self.input_audios[index]['array'])

        mfccs = self.mfcc_transform(input_audio).to(device).transpose(0, 1)

        # Tokenize output
        tokens = torch.tensor(self.tokenizer.tokenize(self.target_sentences[index]), dtype=torch.int32).to(device)

        # Original length
        seq_len = len(mfccs)
        targ_len = len(tokens)

        return mfccs, tokens, seq_len, targ_len


# Collate function
def collate_fn(batch):
    audios, targets, seq_lens, targ_lens = zip(*batch)
    
    audios = pad_sequence(
        audios,
        batch_first=True,
        padding_value=0.0  # Padding for MFCCs
    )

    targets = pad_sequence(
            targets,
            batch_first=True,
            padding_value=tokenizer.vocab['<PAD>']
        )

    return audios, targets, seq_lens, targ_lens

In [57]:
# Mel-frequency ceptral coeffiecients
mfcc_transform = torchaudio.transforms.MFCC(
    sample_rate=16000,
    n_mfcc=13,
    melkwargs={
        "n_fft": 1200,
        "win_length": 1200, # 75ms timeframe
        "hop_length": 600,  
        "n_mels": 40
    }
    )

In [58]:
# Tokenizer
add_blank = ['<Blank>', '<PAD>']
tokenizer = TokenizerBPE01()
tokenizer.loadFromJSON(TOK_PATH)
tokenizer.update(add_blank)
blank_id = tokenizer.vocab['<Blank>']
pad_id = tokenizer.vocab['<PAD>']
print("Blank ID:", blank_id)
print("Pad ID:", pad_id)
print()

Initiate vocabulary of length: 70
Loaded.
Added 2/2 into vocabulary.
Existed: 0/2.
Current vocabulary length: 72
Blank ID: 71
Pad ID: 70



In [59]:
# Make train dataset
input_audios, target_sentences = get_input_and_target(train_set)

train_speech_dataset = SpeechDataset(
    input_audios=input_audios,
    target_sentences=target_sentences,
    tokenizer=tokenizer,
    mfcc_transform=mfcc_transform,
    device=device
)

train_loader = DataLoader(
    train_speech_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

val_audios, val_targets = get_input_and_target(val_set)

val_speech_dataset = SpeechDataset(
    input_audios=val_audios,
    target_sentences=val_targets,
    tokenizer=tokenizer,
    mfcc_transform=mfcc_transform,
    device=device
)

del train_set
del val_set

In [60]:
# Normalizer
class Normalize:
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, x):
        return (x - self.mean) / (self.std + 1e-8)

    def to(self, device):
        self.mean = self.mean.to(device)
        self.std = self.std.to(device)

with open(NORM_PATH, "rb") as f:
    normalizer = pkl.load(f)
normalizer.to(device)
print(normalizer.mean.device)

cpu


In [61]:
# Loss functions
def get_extended_tokens(tokens, blank_id, pad_id, device,):
    # Remove pad_ids
    tokens = tokens[tokens != pad_id]

    # Add blank_ids
    target_length = len(tokens)
    positions = torch.tensor(range(0, 2 * target_length + 1, 2)).to(device)
    extended = torch.empty(2 * target_length + 1, dtype=tokens.dtype).to(device)
    extended[positions] = blank_id
    mask = torch.ones(len(extended), dtype=torch.bool).to(device)
    mask[positions] = False
    extended[mask] = tokens

    return extended

def ctc_loss(logits, targets, seq_lens, blank_id, pad_id, device):
    loss = []
    probabilities = torch.softmax(logits, dim=2)

    for prob, target, seq_len in zip(probabilities, targets, seq_lens):
        # Get extended tokens
        extended = get_extended_tokens(target, blank_id, pad_id, device)
        rev_extended = torch.flip(extended, dims=[0])
        num_states = len(extended)

        # Mask out padded timesteps
        prob = prob[:seq_len]
        time_steps = seq_len

        # Calculate forward-backward
        forwards = []
        forwards_t = torch.cat([
            prob[0, blank_id].unsqueeze(0),
            prob[0, extended[1]].unsqueeze(0),
            prob.new_zeros(num_states - 2)
        ])
        forwards.append(forwards_t)

        backwards = []
        backwards_t = torch.cat([
            prob[-1, blank_id].unsqueeze(0),
            prob[-1, rev_extended[1]].unsqueeze(0),
            prob.new_zeros(num_states - 2)
        ])
        backwards.append(backwards_t)

        for t in range(1, time_steps): 
            forwards_t = torch.zeros(num_states, device=device)
            backwards_t = torch.zeros(num_states, device=device)

            case1f = []
            case2f = []
            
            case1b = []
            case2b = []

            for s in range(num_states):
                # When state in valid range to make it to the end of the sequence for the remaining timesteps
                if (time_steps - t) >= (num_states - s) / 2 - 1:
                    # Case 1:
                    # If <Blank>, prefix must be itself or previous state
                    # If state is duplicated from the non-blank previous state
                    # prefix must be itself or previous state 
                    # If the first state, prefix must be itself or the previous blank state

                    # Case 2:
                    # If state is after the first and is not blank or duplicated / the others,
                    # prefix must be itself, previous state or the state prior to the previous state
                    if extended[s] == blank_id or (s >= 2 and extended[s] == extended[s-2]) or (s == 1):
                        case1f.append(s)
                    else:
                        case2f.append(s)

                    if rev_extended[s] == blank_id or (s >= 2 and rev_extended[s] == rev_extended[s-2]) or (s == 1):
                        case1b.append(s)
                    else:
                        case2b.append(s)
                    
                # When state is at timesteps where it cannot reach the end for the remaining timesteps => 0/default

            case1f = np.asarray(case1f)
            case2f = np.asarray(case2f)

            case1b = np.asarray(case1b)
            case2b = np.asarray(case2b)

            # Case 1
            forw_case1 = prob[t, extended[case1f]] * (forwards[t-1][case1f] + forwards[t-1][case1f-1])
            back_case1 = prob[time_steps-t, rev_extended[case1b]] * backwards[t-1][case1b] + \
                         prob[time_steps-t, rev_extended[case1b-1]] * backwards[t-1][case1b-1]

            # Case 2
            forw_case2 = prob[t, extended[case2f]] * (forwards[t-1][case2f] + forwards[t-1][case2f-1] + forwards[t-1][case2f-2])
            back_case2 = prob[time_steps-t, rev_extended[case2b]] * backwards[t-1][case2b] + \
                         prob[time_steps-t, rev_extended[case2b-1]] * backwards[t-1][case2b-1] + \
                         prob[time_steps-t, rev_extended[case2b-2]] * backwards[t-1][case2b-2]

            # Merge at time step t
            forwards_t = forwards_t.index_copy(0, torch.tensor(case1f, dtype=int, device=device), forw_case1)
            forwards_t = forwards_t.index_copy(0, torch.tensor(case2f, dtype=int, device=device), forw_case2)
            forwards.append(forwards_t)

            backwards_t = backwards_t.index_copy(0, torch.tensor(case1b, dtype=int, device=device), back_case1)
            backwards_t = backwards_t.index_copy(0, torch.tensor(case2b, dtype=int, device=device), back_case2)
            backwards.append(backwards_t)


        forwards = torch.stack(forwards)
        backwards = torch.stack(backwards)

        # Calculate probability of sequence
        backwards = torch.flip(backwards, dims=[1])
        backwards = torch.flip(backwards, dims=[0])
        yammas = forwards * backwards
        log_p_seqs = torch.log((yammas + 1e-8) / (prob[:, extended] + 1e-8)).mean(dim=1)

        #Calculate loss
        l = - log_p_seqs.mean()
        loss.append(l)

    loss = torch.stack(loss).mean().to(device)
    return loss

In [62]:
# Model
class SimpleRNN(nn.Module):
    def __init__(
        self,
        input_size,
        output_size,
        hidden_dim
    ):
        super().__init__()

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_dim,
            output_size
        )

    def forward(self, X):
        output, _ = self.rnn(X)

        return self.fc(output)

model = SimpleRNN(13, len(tokenizer.vocab), 16).to(device)

In [63]:
# Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

In [64]:
for epoch in range(1, EPOCHS+1):
    print(f"[Epoch: {epoch}/{EPOCHS}]")
    # Train on random a small proportion of trainning data
    max_batches = int(len(train_loader) * BATCH_RATIO)
    # Train
    model.train()
    full_set_loss = []
    for i, batch in enumerate(train_loader):
        if i >= max_batches:
            break

        x, y, seq_len, targ_len = batch
        optimizer.zero_grad()
        logits = model(normalizer(x))
        log_probs = torch.log_softmax(logits, dim = -1).transpose(0, 1)

        loss = ctc_loss(logits, y, seq_len, blank_id, pad_id, device)
        loss.backward()
        # # Check grad
        # for name, param in model.named_parameters():
        #     if param.grad is not None:
        #         print(f"{name} grad:\n{param.grad}\n")
        #     else:
        #         print(f"{name} grad: None (no gradient computed)\n")
        optimizer.step()
        full_set_loss.append(loss.item())
        print(f"Batch {i} loss: {loss.item()}")
       
    train_loss = float(np.mean(full_set_loss))
    print(f"Train loss     : {train_loss}")

    # Evaluate
    model.eval()
    full_set_loss = []
    with torch.no_grad():
        for x, y, seq_len, targ_len in val_speech_dataset:
            logits = model(normalizer(x).unsqueeze(0))
            log_probs = torch.log_softmax(logits, dim = -1).transpose(0, 1)
            loss = ctc_loss(logits, y.unsqueeze(0), (seq_len,), blank_id, pad_id, device)
            #loss = criterion(log_probs, y.unsqueeze(0), torch.tensor([seq_len,], dtype=int, device=device), torch.tensor([targ_len], dtype=int, device=device))
            full_set_loss.append(loss.item())
        # Check single output
        with torch.no_grad():
            output, prob = beam_search(logits, 10, blank_id)
            print(prob)
        print(tokenizer.tokenToWords(collpase(output[0], blank_id)))

    val_loss = float(np.mean(full_set_loss))
    print(f"Validation loss: {val_loss}")

[Epoch: 1/3]
Batch 0 loss: 14.055793762207031
Batch 1 loss: 14.050203323364258
Batch 2 loss: 14.045154571533203
Batch 3 loss: 14.03270149230957
Batch 4 loss: 14.023763656616211
Train loss     : 14.041523361206055
0.100017086
}_g_g_.5;.g6.5;5g;_.5!.;.;.!5._!._._.*,}85.g5!5.5.g!._._gq5g5x^3^!.!.5.5m8_._.;._g85;xg._.!5!.^qx;_^}85.5x_.!.;}x}x,_3{_{_,_#g85g5xg{g_g8g5x._,g3g^5g.`:5.!_g8x_<g_g_}g_g{_,:g5x.5.x5x.!.qg}xg5x5g{g._g8._3<PAD>8qg.5xg{g.5;!._}<^;^
Validation loss: 13.93494910925207
[Epoch: 2/3]
Batch 0 loss: 14.015966415405273
Batch 1 loss: 13.996965408325195
Batch 2 loss: 13.994327545166016
Batch 3 loss: 13.976663589477539
Batch 4 loss: 13.973687171936035
Train loss     : 13.991522026062011
0.100085475
}_g{g_.5;.g6.5;5g;_.!.;.;.!5._!._._.*,}8q5.g5!5.5.g{g!.3.gq5g5x^3;!.!.5m8;._.;._g85;.xg.!8.^qx;_^}85.5x.;.x.}x_3{_g{_,{,_#gq5g5xg{g_g8g5x._g^5g.`:5.!_._g8x_.<{_g_}g_g{:g5x.x5x.!.qg}xg5x5g:g{g._g8._3<PAD>8qg.5xg{g.;!._}<g;^
Validation loss: 13.857449934516154
[Epoch: 3/3]
Batch 0 loss: